In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Library

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# Utils

In [3]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

#  Sample Clothing and Shoes

In [7]:
# 리뷰 데이터셋 확인
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_final.pkl')
df_sample = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_simple_v5_final.pkl')

print(df.shape)
df.head()

(66718, 17)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06857138, 0.13921358, -0.039587017, -0.088...","Based on user reviews, it appears that users a...",The image shows a hand holding a set of earrin...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.04617965, 0.12618569, -0.017711096, -0.102...","Based on user reviews, it appears that users p...",The image shows a close-up of a person's ear w...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.062135138, 0.1333665, -0.019405615, -0.102...","Based on this review, it seems that users pref...","The image shows a fashion accessory, specifica..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06780286, 0.051634714, -0.017799914, -0.11...","Based on user reviews, it appears that users p...",The image shows a sleeveless tank top with a p...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.07941287, 0.05176169, -0.02554227, -0.0903...","Based on the user review, it appears that the ...",The image shows a lime green t-shirt with a gr...


In [8]:
# 복원을 위한 Column 선별

# 6만 7천개
df = df[['user_id', 'asin', 'parent_asin', 'text', 'rating', 'title_y', 'small_image_url', 'medium_image_url', 'large_image_url']]

# 3만개
df_sample = df_sample[['text', 'LLaMA_Output', 'LLaVA_Output']]

In [18]:
df_diff = df.merge(df_sample, how='left', left_on='text', right_on='text')
df_diff = df_diff[df_diff['LLaMA_Output'].isna()]
df_diff = df_diff.drop(columns=['LLaMA_Output','LLaVA_Output'])
df_diff = df_diff.reset_index(drop=True)

print(df_diff.shape)
df_diff.head()

(36856, 9)


,user_id,asin,parent_asin,text,rating,title_y,small_image_url,medium_image_url,large_image_url
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B074BW37RT,B085Y99HGC,Great quality necklace. Looks exactly like the...,5,Pearl Necklace Genuine Freshwater Cultured 9-1...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
3,AFXF3EGQTQDXMRLDWFU7UBFQZB7Q,B08TSXWLGJ,B08TSXWLGJ,There are six beautiful headwraps in this set....,5,Preknotted African Pattern Headwrap -6 Pieces ...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
4,AFXF3EGQTQDXMRLDWFU7UBFQZB7Q,B088NJWRBV,B088NJWRBV,My first reaction to this set of three socks w...,5,iCamter 10 Pairs Womens No Show Socks Non Slip...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...


In [19]:
df.shape

(66718, 9)

In [20]:
df_sample.shape, df_diff.shape

((29862, 3), (36856, 9))

In [21]:
df_diff.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_RQ3_diff.pkl')